In [13]:
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.utils.data as data
import wandb
import os
import umap
import matplotlib.pyplot as plt
import numpy as np


In [14]:
!pip install lightly

In [15]:
from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform

In [39]:
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/ProjectSimCLR'
else:
    # Lokal auf deinem MacBook
    # Nutze absolute Pfade oder relative Pfade zu deinem Repository
    base_path = './'

def save_checkpoint(model, optimizer, epoch, loss, run_id):
    torch.save({"epoch": epoch,
    "optimizer":optimizer,
    "model": model,
    "loss": loss,
    "wandb_run_id": run_id,
                "hp" : hp
    }, os.path.join(base_path, f"{run_id}_epoch_{epoch}.pth"))

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint['loss'], checkpoint['wandb_run_id']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
class SimCLR(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, 1)
        x = self.projection_head(x)
        return x


In [18]:
resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(
    input_dim = 512,
    hidden_dim=512,
    output_dim=128
)
model = SimCLR(backbone, projection_head)

In [19]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)

SimCLR(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [20]:
hp = {
    "batch_size": 128,
    "pipeline_test" : False,
    "epochs":50,
    "optim" : {
        "lr" : 0.06,
        "weight_decay": 1e-5
    }
    }

In [21]:
transform = SimCLRTransform(input_size=32, gaussian_blur=0.0)
dataset = datasets.CIFAR10(root="./data", train=True, transform=transform,  download=True)
if hp["pipeline_test"]:
    dataset = data.Subset(dataset, range(256))

In [22]:
dataloader = DataLoader(dataset, batch_size=hp["batch_size"], shuffle=True, drop_last=True,num_workers=2)


In [23]:
criterion = NTXentLoss()
optimizer = torch.optim.SGD(params=model.named_parameters(), **hp["optim"])

In [24]:
print("Starting Training ...")
run = wandb.init(project="simclr-cifar10", config=hp)
model = model.to(device)
for epoch in range(hp["epochs"]):
    print(f"Epoch {epoch}")
    total_loss = 0
    for batch in dataloader:
        x0, x1 = batch[0]
        x0 = x0.to(device)
        x1 = x1.to(device)
        z0 = model(x0)
        z1 = model(x1)

        loss = criterion(z0, z1)
        total_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    avg_loss = total_loss / len(dataloader)
    wandb.log({
    "loss": avg_loss,
    "epoch": epoch,
    "lr": optimizer.param_groups[0]['lr']
    })
    if epoch % 1 == 0:
        save_checkpoint(model, optimizer, epoch, loss, run.id)
        print(f"Saved model after epoch: {epoch}")
    print(f"Epoch: {epoch}, Avg_loss: {avg_loss:.5f}")

Starting Training ...


epoch,▁
loss,▁
lr,▁
epoch,0
loss,5.11198
lr,0.06


Epoch 0
Epoch: 0, Avg_loss: 5.11713
Epoch 1
Epoch: 1, Avg_loss: 4.92329
Epoch 2
Epoch: 2, Avg_loss: 4.84042
Epoch 3
Epoch: 3, Avg_loss: 4.78581
Epoch 4
Epoch: 4, Avg_loss: 4.74973
Epoch 5
Epoch: 5, Avg_loss: 4.72677
Epoch 6
Epoch: 6, Avg_loss: 4.70174
Epoch 7
Epoch: 7, Avg_loss: 4.68416
Epoch 8
Epoch: 8, Avg_loss: 4.66444
Epoch 9
Epoch: 9, Avg_loss: 4.66257
Epoch 10
Epoch: 10, Avg_loss: 4.64282
Epoch 11
Epoch: 11, Avg_loss: 4.63126
Epoch 12
Epoch: 12, Avg_loss: 4.62260
Epoch 13
Epoch: 13, Avg_loss: 4.61795
Epoch 14
Epoch: 14, Avg_loss: 4.60286
Epoch 15
Epoch: 15, Avg_loss: 4.59900
Epoch 16
Epoch: 16, Avg_loss: 4.58984
Epoch 17
Epoch: 17, Avg_loss: 4.58359
Epoch 18
Epoch: 18, Avg_loss: 4.57793
Epoch 19
Epoch: 19, Avg_loss: 4.57358
Epoch 20


TypeError: save_checkpoint() missing 1 required positional argument: 'run_id'

In [25]:
import umap
import matplotlib.pyplot as plt
import numpy as np


In [40]:
save_checkpoint(model, optimizer, epoch, loss, run.id)

In [33]:
model.eval()
embeddings = []
labels = []

with torch.no_grad():
    for imgs, targets in dataloader:
        z = model.encoder(imgs[0].to(device))
        embeddings.append(z.cpu().numpy())
        labels.append(targets.numpy())

embeddings = np.concatenate(embeddings)
labels = np.concatenate(labels)

reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1],
    c=labels,
    cmap='tab10',
    alpha=0.5,
    s=5
)
plt.colorbar(scatter)
plt.title('UMAP of SimCLR Embeddings (no projection head)')
plt.savefig('umap.png', dpi=150)
wandb.log({"umap": wandb.Image("umap.png")})

ValueError: Found array with dim 4. None expected <= 2.